# FDSI benchmark v1.1: fixed (no-adaptation) baseline

The v1.1 counterpart of `../fdsi_benchmark/02_fixed_adaptation.ipynb`. Same calibrations, same
grid, same no-adaptation config; the one difference is `process_from_calib_end`, which keeps
CBSS's own output over the calibration window and starts the online pass at its end.

## Config

In [ ]:
RUN_DECOMP_FIXED = False  # compute missing adaptation_v1_1/<sub>/fixed/..._adapt_fixed.pkl

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

DATA_DIR = Path(grid['data_root'])           # Raw archive root: <sub>/{clean,noisy}/
OUTPUTS_ROOT = Path(grid['outputs_root'])    # Precomputed-outputs archive root
CAL_DIR = OUTPUTS_ROOT / 'calibration'       # Shared with v1.0 -- written by ../fdsi_benchmark/01_calibration
ADAPT_DIR = OUTPUTS_ROOT / 'adaptation_v1_1' # v1.1 results
ADAPT_DIR_V10 = OUTPUTS_ROOT / 'adaptation'  # Cached v1.0 results, read-only here
ADAPT_DIRS = {'v1.0': ADAPT_DIR_V10, 'v1.1': ADAPT_DIR}
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_PATH = Path('../../configs/adapt_configs/default_muniverse.yaml')  # v1.0's base config
LR_MODE = 'fixed'   # v1.1 searches lr_fixed only -- the branch v1.0 promoted as its headline
DEVICE = 'cpu'      # deterministic, and MPS lacks the QR the sv update needs

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')
print(f'Calibration window [0, {CAL_END}) samples; adaptation starts at sample {CAL_END}')

## Calibration

v1.1 reuses the calibrations `../fdsi_benchmark/01_calibration.ipynb` wrote, so that any v1.0 vs
v1.1 difference comes from the adaptation stage alone. Nothing is recomputed here.

In [ ]:
import sys
sys.path.insert(0, '../..')          # package root
sys.path.insert(0, '../fdsi_benchmark')  # for fdsi_common, shared with the v1.0 benchmark
sys.path.insert(0, str(Path.cwd()))  # this notebook's own directory, for fdsi_v11

from typing import Dict, Optional, Tuple
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

import numpy as np
from adapt_decomp import CBSSResult
import fdsi_common as fc
import fdsi_v11 as fv

calibrations: Dict[Tuple[str, str, int], Optional[CBSSResult]] = {}
for sub in SUBJECTS:
    for cond in CONDITIONS:
        for snr in SNR_LEVELS:
            cal_path, _ = fc.calibration_paths(CAL_DIR, sub, cond, snr)
            calibrations[(sub, cond, snr)] = CBSSResult.load(cal_path) if cal_path.exists() else None

n_cal = sum(v is not None for v in calibrations.values())
print(f'Calibration available: {n_cal}/{len(calibrations)} recordings.')

## Run the no-adaptation baseline on one recording

In [ ]:
from adapt_decomp import AdaptDecomp, CBSSConfig
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.spikes import get_sil

sub, cond, snr = OPTIM_SUB, TRIANGULAR[0], OPTIM_SNR
cbss_result = calibrations[(sub, cond, snr)]

# Read back this recording's own CBSSConfig -- ext_fact etc. must match the calibration it came from
emg_full = fc.load_raw_emg(DATA_DIR, sub, cond, snr)
_, cal_config_path = fc.calibration_paths(CAL_DIR, sub, cond, snr)
cbss_config = CBSSConfig.from_yaml(cal_config_path)

# Every adaptation flag off -- the reference floor both versions are measured against
adapt_config = AdaptConfig(ext_fact=EXT_FACT, adapt_wh=False, adapt_sv=False, adapt_sd=False,
                           device=DEVICE)
adapter = AdaptDecomp.from_calibration(calibration=cbss_result, cbss_config=cbss_config,
                                        adapt_config=adapt_config)

# v1.1: keep CBSS's output over [0, CAL_END) and run forwards from CAL_END
outputs = adapter.process_from_calib_end(emg_full, slice(0, CAL_END))

# Compute SIL and RoA for the full recording
outputs.sil = get_sil(outputs.sources, outputs.spikes, adapt_config.spike_min_dist,
                       peak_power=adapt_config.spike_det_exp).numpy()
gt_full_bin = fc.load_gt_full_bin(DATA_DIR, sub, cond, cbss_result,
                                   n_samples=outputs.spikes.shape[0])
outputs.roa = fc.compute_roa_for_result(outputs, gt_full_bin, FS, TOL_SPIKE_MS)

print(f'{sub} {cond} snr{snr}dB: {int(outputs.spikes.sum())} spikes over '
      f'{outputs.spikes.shape[0] / FS:.1f} s (no adaptation).')

### Execution time -- v1.0 vs v1.1

In [ ]:
from adapt_decomp import AdaptationResult

timings = {
    'v1.1': outputs,
    'v1.0': AdaptationResult.load(fc.fixed_paths(ADAPT_DIR_V10, sub, cond, snr)[0]),
}

title = 'No adaptation execution time (ms per batch)'
print(title + '\n' + '-' * len(title))
for key in ['preprocess_time_ms', 'wh_time_ms', 'sv_time_ms', 'sd_time_ms', 'total_time_ms']:
    print(f'{key:>19}: ' + '   '.join(f'{label} {out[key].mean():.3f} +/- {out[key].std():.3f}'
                                       for label, out in timings.items()))

In [ ]:
import matplotlib.pyplot as plt

from adapt_decomp.utils.plots import plot_spikes

ts_full = np.arange(outputs.spikes.shape[0]) / FS
ax = plot_spikes(
    spikes={**{f'No adaptation ({label})': out.spikes.numpy() for label, out in timings.items()},
            'Ground truth': gt_full_bin},
    timestamps=ts_full,
    time_range=(0, ts_full[-1]),
    roa={f'No adaptation ({label})': out.roa for label, out in timings.items()},
    sil={f'No adaptation ({label})': out.sil for label, out in timings.items()},
)
ax.axvline(CAL_END / FS, color='k', ls='--', lw=1)
ax.set_title(f'Output spikes vs ground truth -- {sub} {cond} snr{snr}dB '
             f'(no adaptation; dashed: calibration end)')
plt.show()

## Apply the no-adaptation baseline to the full benchmark grid

In [ ]:
from joblib import Parallel, delayed

from adapt_decomp import AdaptDecomp, AdaptationResult, CBSSConfig
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.spikes import get_sil


def run_adapted(sub: str, cond: str, snr: int, cbss_result: Optional[CBSSResult],
                adapt_config: AdaptConfig, sampler: Optional[str]) -> Optional[AdaptationResult]:
    """Load this recording's cached v1.1 result, or compute it if RUN_DECOMP_FIXED is True.

    Same steps as v1.0's run_adapted, except process_from_calib_end replaces process_data:
    the calibration window [0, CAL_END) keeps CBSS's own output and adaptation runs forwards
    from CAL_END, so the adapted segment is never scored against a window it was fitted on.

    Args:
        sub (str): Subject id.
        cond (str): Condition name.
        snr (int): SNR level in dB.
        cbss_result (Optional[CBSSResult]): This recording's calibration, or None if
            calibration itself was skipped.
        adapt_config (AdaptConfig): The config to apply.
        sampler (Optional[str]): v1.1 sampler dir, or None for the fixed baseline.

    Returns:
        Optional[AdaptationResult]: The result, or None if uncached/uncomputable.
    """
    if cbss_result is None:
        return None
    lr_mode = None if sampler is None else LR_MODE
    result_path, config_path = fv.result_paths(ADAPT_DIR, sub, cond, snr, lr_mode, sampler)
    if result_path.exists():
        return AdaptationResult.load(result_path)
    if not RUN_DECOMP_FIXED:
        return None

    # Read back this recording's own CBSSConfig -- ext_fact etc. must match its calibration
    emg_full = fc.load_raw_emg(DATA_DIR, sub, cond, snr)
    _, cal_config_path = fc.calibration_paths(CAL_DIR, sub, cond, snr)
    cbss_config = CBSSConfig.from_yaml(cal_config_path)

    # Initialise adaptation from calibration and adapt forwards from the calibration end
    adapter = AdaptDecomp.from_calibration(calibration=cbss_result, cbss_config=cbss_config,
                                            adapt_config=adapt_config)
    outputs = adapter.process_from_calib_end(emg_full, slice(0, CAL_END))

    # Compute SIL and RoA for the full recording
    outputs.sil = get_sil(outputs.sources, outputs.spikes, adapt_config.spike_min_dist,
                           peak_power=adapt_config.spike_det_exp).numpy()
    gt_full_bin = fc.load_gt_full_bin(DATA_DIR, sub, cond, cbss_result,
                                       n_samples=outputs.spikes.shape[0])
    outputs.roa = fc.compute_roa_for_result(outputs, gt_full_bin, FS, TOL_SPIKE_MS)

    # Save the adaptation result and config
    result_path.parent.mkdir(parents=True, exist_ok=True)
    outputs.save(result_path)
    adapt_config.to_yaml(config_path)
    return outputs


fixed_config = AdaptConfig(ext_fact=EXT_FACT, adapt_wh=False, adapt_sv=False, adapt_sd=False,
                           device=DEVICE)

N_JOBS = 4  # Recordings run in parallel, one per worker process

keys = list(calibrations)
jobs = [(sub, cond, snr, calibrations[(sub, cond, snr)], fixed_config, None)
        for (sub, cond, snr) in keys]
pbar = tqdm(total=len(jobs), desc='Apply fixed (v1.1)')
fixed_results: Dict[Tuple[str, str, int], Optional[AdaptationResult]] = {}
results_iter = Parallel(n_jobs=N_JOBS, return_as='generator')(delayed(run_adapted)(*job) for job in jobs)
for key, result in zip(keys, results_iter):
    fixed_results[key] = result
    pbar.update(1)
pbar.close()

n_fixed = sum(v is not None for v in fixed_results.values())
print(f'Fixed baseline available: {n_fixed}/{len(fixed_results)} recordings.')

## v1.0 vs v1.1 baseline over the grid

Scored twice: over the full recording, and over `[CAL_END, end)` only. The windowed table is
the fair one -- inside the calibration window v1.1 emits CBSS's own spikes while v1.0 re-adapted
over it, so a full-recording difference partly measures that bookkeeping rather than adaptation.

In [ ]:
import pandas as pd

CONFIG_LABELS = fv.configs()   # just the two fixed baselines

df_roa = fv.roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                      CONFIG_LABELS, FS, TOL_SPIKE_MS)
df_post = fv.window_roa_table(ADAPT_DIRS, CAL_DIR, DATA_DIR, SUBJECTS, CONDITIONS, SNR_LEVELS,
                              CONFIG_LABELS, slice(CAL_END, None), FS, TOL_SPIKE_MS)

if len(df_roa) and len(df_post):
    display(pd.concat({'full recording': fv.paired_summary(df_roa, fv.pairs()),
                       'after calibration': fv.paired_summary(df_post, fv.pairs())}).round(3))
else:
    print('Nothing cached yet for one of the two versions.')

### Reading this

`fixed_results` is this notebook's own in-memory result; `03*` and `04*` rebuild what they need
from the same cache, so the notebooks can be run in any order once the cache is warm.